# Clusterless High-D Decoding - Quickstart

This notebook demonstrates basic usage of the `clusterless_high_d` package for decoding hippocampal datasets.

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

# Package imports
from clusterless_high_d import run_decode, load_dataset_registry, load_dataset

%matplotlib inline
%load_ext autoreload
%autoreload 2

## 1. View available datasets

In [ ]:
# Load the dataset registry
registry = load_dataset_registry("../configs/datasets.yaml")

print("Available datasets:")
for dataset_id, info in registry.items():
    print(f"\n{dataset_id}:")
    print(f"  Region: {info['region']}")
    print(f"  Contacts: {info['n_contacts']}")
    print(f"  Description: {info['description']}")

## 2. Load configuration

In [ ]:
# Load base configuration
with open("../configs/base.yaml") as f:
    config = yaml.safe_load(f)

print("Decoding configuration:")
print(yaml.dump(config, default_flow_style=False))

## 3. Run decoding for a single dataset

In [ ]:
# Choose a dataset
dataset_id = "rat01_day1"

# Run decoding
result = run_decode(
    dataset_id=dataset_id,
    config_path=Path("../configs/base.yaml"),
    plots_config_path=Path("../configs/plots.yaml"),
    outdir=Path("../results"),
)

print("\nDecoding result:")
for key, value in result.items():
    print(f"  {key}: {value}")

## 4. Load and visualize results

In [ ]:
# Load the saved posterior
output_dir = result["output_dir"]
data = np.load(output_dir / "posterior.npz")

time = data["time"]
posterior = data["posterior"]
actual_position = data["actual_position"]

print(f"Loaded data shapes:")
print(f"  time: {time.shape}")
print(f"  posterior: {posterior.shape}")
print(f"  actual_position: {actual_position.shape}")

In [ ]:
# Compute decoded position (argmax)
n_time = len(time)
decoded_position = np.zeros((n_time, 2))

for t in range(n_time):
    flat_idx = posterior[t].argmax()
    decoded_position[t, 0] = flat_idx // posterior.shape[2]
    decoded_position[t, 1] = flat_idx % posterior.shape[2]

# Normalize to match actual position scale
decoded_position[:, 0] = (
    decoded_position[:, 0] / posterior.shape[1] * actual_position[:, 0].max()
)
decoded_position[:, 1] = (
    decoded_position[:, 1] / posterior.shape[2] * actual_position[:, 1].max()
)

In [ ]:
# Plot trajectory comparison
fig, ax = plt.subplots(figsize=(10, 10))

ax.plot(
    actual_position[:, 0],
    actual_position[:, 1],
    "r-",
    alpha=0.5,
    linewidth=2,
    label="Actual",
)
ax.plot(
    decoded_position[:, 0],
    decoded_position[:, 1],
    "b-",
    alpha=0.5,
    linewidth=2,
    label="Decoded",
)

ax.set_xlabel("X Position")
ax.set_ylabel("Y Position")
ax.set_title(f"Trajectory: {dataset_id}")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
# Plot position error over time
position_error = np.sqrt(
    (actual_position[:, 0] - decoded_position[:, 0]) ** 2
    + (actual_position[:, 1] - decoded_position[:, 1]) ** 2
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time, position_error, "k-", linewidth=1)
ax.axhline(
    position_error.mean(),
    color="r",
    linestyle="--",
    label=f"Mean: {position_error.mean():.2f}",
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Position Error")
ax.set_title("Decoding Error Over Time")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nError statistics:")
print(f"  Mean: {position_error.mean():.2f}")
print(f"  Median: {np.median(position_error):.2f}")
print(f"  Std: {position_error.std():.2f}")

## 5. View generated plots

The decode function automatically generates plots in the output directory.

In [ ]:
# List generated plots
plots_dir = output_dir / "plots"
plot_files = list(plots_dir.glob("*.png"))

print(f"Generated plots in {plots_dir}:")
for plot_file in plot_files:
    print(f"  - {plot_file.name}")

In [ ]:
# Display generated plots
from IPython.display import Image, display

for plot_file in sorted(plot_files):
    print(f"\n{plot_file.name}:")
    display(Image(filename=plot_file))

## 6. Explore configuration options

In [ ]:
# Example: modify config for a custom run
custom_config = config.copy()
custom_config["decoding"]["time_window"] = 1.0  # Increase time window
custom_config["high_dimensional"]["use_sparse"] = False  # Disable sparse mode

print("Custom configuration:")
print(yaml.dump(custom_config, default_flow_style=False))

# Save custom config
custom_config_path = Path("../configs/custom.yaml")
with open(custom_config_path, "w") as f:
    yaml.dump(custom_config, f, default_flow_style=False)

print(f"\nSaved to: {custom_config_path}")

## 7. Next steps

- Modify `configs/base.yaml` to adjust decoding parameters
- Modify `configs/plots.yaml` to customize visualizations
- Add more datasets to `configs/datasets.yaml`
- Use the CLI for batch processing: `uv run python -m clusterless_high_d batch`
- Implement actual Spyglass queries in `src/clusterless_high_d/spyglass_io.py`
- Replace placeholder decoder with `non_local_detector` in `src/clusterless_high_d/decode.py`